In [1]:
from Data_polisher import DataPolish
import pandas as pd
from catboost import CatBoostClassifier
from ModelTrainer import ModelTrainer
from sklearn.model_selection import train_test_split
from Hypotheses import Hypothesis1
import matplotlib.pyplot as plt


ImportError: cannot import name 'Hypothesis1' from 'Hypotheses' (c:\Users\klmv0\OneDrive\Desktop\project_2_ml\Hypotheses.py)

In [ ]:
train = pd.read_csv('train_final_processed.csv')
test = pd.read_csv('test_final_processed.csv')

In [ ]:
y = train['target']
X_raw = train.drop(columns=['target', 'sk_id_curr'])
X_test_raw = test.drop(columns=['sk_id_curr'])
test_ids = test['sk_id_curr']

print(f"Исходно: {X_raw.shape[1]} фич")

Исходно: 569 фич


In [ ]:
hypothesis1 = Hypothesis1()
X_with_h1 = hypothesis1.fit_transform(X_raw)
X_test_with_h1 = hypothesis1.transform(X_test_raw)

✅ Гипотеза 1 (Installments): +7 фич → ['ip_underpayment_total', 'ip_underpayment_ratio', 'ip_paid_ratio', 'ip_max_delay', 'ip_avg_delay', 'ip_restructured_flag', 'ip_version_diversity']
✅ Гипотеза 1 (Installments): +7 фич → ['ip_underpayment_total', 'ip_underpayment_ratio', 'ip_paid_ratio', 'ip_max_delay', 'ip_avg_delay', 'ip_restructured_flag', 'ip_version_diversity']


In [ ]:
new_features = [col for col in X_with_h1.columns if col not in X_raw.columns]
print(f"Добавленные фичи: {new_features}")

Добавленные фичи: ['ip_underpayment_total', 'ip_underpayment_ratio', 'ip_paid_ratio', 'ip_max_delay', 'ip_avg_delay', 'ip_restructured_flag', 'ip_version_diversity']


In [ ]:
if new_features:
    # Создаем временный DataFrame для анализа
    temp_df = X_with_h1[new_features].copy()
    temp_df['target'] = y.values
    
    print("Статистика новых фич:")
    for feature in new_features:
        print(f"\n--- {feature} ---")
        print(f"   Min: {temp_df[feature].min():.4f}")
        print(f"   Max: {temp_df[feature].max():.4f}")
        print(f"   Mean: {temp_df[feature].mean():.4f}")
        print(f"   Std: {temp_df[feature].std():.4f}")
        print(f"   NaN: {temp_df[feature].isnull().sum()} ({temp_df[feature].isnull().mean()*100:.1f}%)")
        
        # Корреляция с целевой переменной
        if temp_df[feature].nunique() > 1:
            corr = temp_df[feature].corr(temp_df['target'])
            print(f"   Corr with target: {corr:.4f}")

Статистика новых фич:

--- ip_underpayment_total ---
   Min: -4417384.2300
   Max: 3037735.5750
   Mean: -5851.1481
   Std: 170720.5708
   NaN: 0 (0.0%)
   Corr with target: 0.0272

--- ip_underpayment_ratio ---
   Min: -4.4851
   Max: 1.0000
   Mean: 0.0078
   Std: 0.1327
   NaN: 0 (0.0%)
   Corr with target: 0.0523

--- ip_paid_ratio ---
   Min: 0.0000
   Max: 5.4851
   Mean: 0.9406
   Std: 0.2564
   NaN: 0 (0.0%)
   Corr with target: -0.0116

--- ip_max_delay ---
   Min: 0.0000
   Max: 4906.0000
   Mean: 1214.7832
   Std: 962.1304
   NaN: 0 (0.0%)
   Corr with target: -0.0492

--- ip_avg_delay ---
   Min: -2993.0000
   Max: 0.0000
   Mean: -894.3149
   Std: 602.1624
   NaN: 0 (0.0%)
   Corr with target: 0.0352

--- ip_restructured_flag ---
   Min: 0.0000
   Max: 1.0000
   Mean: 0.0116
   Std: 0.1073
   NaN: 0 (0.0%)
   Corr with target: 0.0028

--- ip_version_diversity ---
   Min: 0.0000
   Max: 53.0000
   Mean: 2.1579
   Std: 1.8380
   NaN: 0 (0.0%)
   Corr with target: -0.0048


In [ ]:
polisher = DataPolish(
    max_na_ratio=0.7,
    corr_threshold=0.95,
    handle_outliers=True,
    outlier_method='clip',
    log_transform_skew=1.5,
    scale_numeric=False,
    verbose=True
)

X_clean = polisher.fit_transform(X_with_h1)
X_test_clean = polisher.transform(X_test_with_h1)

print(f"После очистки: {X_clean.shape[1]} фич")


=== ОЧИСТКА ДАННЫХ ===


MemoryError: Unable to allocate 1.05 GiB for an array with shape (458, 307511) and data type float64

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_clean, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [ ]:
categorical_columns = X_clean.select_dtypes(include=['object', 'category']).columns.tolist()

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='SqrtBalanced',
    random_state=42,
    verbose=100,
    early_stopping_rounds=200,
    eval_metric='AUC',
    cat_features=categorical_columns,
    thread_count=-1
)

trainer = ModelTrainer(model, "HYPOTHESIS8_TEST")
trainer.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

In [ ]:
try:
    # Получаем важность фич
    feature_importance = trainer.model.get_feature_importance()
    feature_names = X_clean.columns
    
    # Создаем DataFrame с важностью
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
  

    print(importance_df.head(20).to_string(index=False))
    
    # Проверяем позиции новых фич
    new_features_importance = importance_df[importance_df['feature'].isin(new_features)]
    print(f"\n Позиции новых фич из Hypothesis1:")
    for idx, row in new_features_importance.iterrows():
        rank = importance_df.index.get_loc(idx) + 1
        print(f"   #{rank:2d}: {row['feature']} - {row['importance']:.4f}")
    
    # Визуализация
    plt.figure(figsize=(12, 8))
    
    # Топ-15 фич
    top_features = importance_df.head(15)
    
    # Разделяем на новые и старые фичи для цветового кодирования
    colors = ['red' if feat in new_features else 'blue' for feat in top_features['feature']]
    
    plt.barh(range(len(top_features)), top_features['importance'], color=colors)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Важность фичи')
    plt.title('ТОП-15 самых важных фич\nКрасные - новые из Hypothesis1')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('hypothesis1_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
except Exception as e:
    print(f" Ошибка при анализе важности фич: {e}")

In [ ]:
try:
    val_proba = trainer.model.predict_proba(X_val)[:, 1]
    trainer.plot_curves(y_val, val_proba)
    print("Графики ROC и Precision-Recall сохранены")
except Exception as e:
    print(f"Ошибка при построении графиков: {e}")

In [ ]:
# 10. ПРЕДСКАЗАНИЕ НА ТЕСТЕ
try:
    proba, _ = trainer.predict_test(X_test_clean)
    
    # Создаем сабмит
    sub = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': proba})
    sub.to_csv('submission_hypothesis8_only.csv', index=False)
    
except Exception as e:
    print(f"Ошибка при создании сабмита: {e}")